# Lakshminarasimhan et al. (2024) on dandiset 000009

**Paper.** Lakshminarasimhan KJ, Xie M, Cohen JD, Sauerbrei BA, Hantman AW, Litwin-Kumar A,
Escola S (2024). Specific connectivity optimizes learning in thalamocortical loops. *Cell Reports*
43, 114059. [doi:10.1016/j.celrep.2024.114059](https://doi.org/10.1016/j.celrep.2024.114059). The
claims file (`claims/lakshminarasimhan-2024.yaml`) was written from the NIH author manuscript in
PubMed Central (PMC11104520) with the supplement (mmc2).

**Dandiset.** [DANDI:000009](https://dandiarchive.org/dandiset/000009/0.220126.1903), version
0.220126.1903, "Maintenance of persistent activity in a frontal thalamocortical loop" (Guo et al.
2017). The paper cites the dandiset without a version (key resources table). This is the only
published version, and it is pinned for that reason. The draft holds the same 173 files with the
same blobs (checked below); it differs only in metadata.

**What the paper used from it.** The working memory data of Figure 6 and Figure S6: "5 behavioral
sessions from one mouse", 151 ALM and 72 thalamic (VM/VAL) units, correct trials, delay period. The
motor control data of the same figures (Sauerbrei et al. 2020) are not in this dandiset, nor in
any archive the paper names; a search of the DANDI API for "Sauerbrei" finds only dandiset 000221,
a different study. The simulations of Figures 1 to 5 and S1 to S5 are generated by the paper's
models and are not deposited.

**Method.** Each claim's criterion refers to the numbered method notes W1 to W10 at the head of the
claims file; the helper `tcloop.py` says for each parameter where it comes from. Random splits and
subsamples use each claim's own seed (`tcloop.seed`), and each result that depends on them is also
computed under the seed plus 1000, 2000, 3000 and 4000.

| Claim | Data | Status |
|---|---|---|
| wm-sessions-link | dandiset | tested |
| fig6-wm-choice-r2 | dandiset | tested |
| fig6c-wm-leading-pcs-drive-choice | dandiset | tested |
| figs6d-wm-thalamic-r2 | dandiset | tested |
| fig6d-wm-top5-pcs | dandiset | tested |
| figs6f-ct-alignment-not-significant | dandiset | tested |
| fig6-motor-hand-r2, fig6c-motor-low-variance-pcs, figs6d-motor-thalamic-r2, fig6d-motor-top5-pcs, fig6e-motor-single-trial | dandiset | blocked: data missing (Sauerbrei et al. 2020 not deposited) |
| fig6d-more-pcs-in-motor-task | dandiset | blocked: data missing (needs the motor data) |
| fig1c to figs5j (24 simulation claims) | elsewhere | blocked: too large (timed below), except figs5cd-thalamic-recurrence: methods underspecified |

In [ ]:
import time
import resource
import hashlib
import os
import warnings
from collections import Counter

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from scipy import stats

import tcloop as tc

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
warnings.filterwarnings("ignore", category=RuntimeWarning)
started = time.time()
print("dandiset", tc.DANDISET, "version", tc.VERSION)

## The dandiset

Every file is listed from the archive, and every extracellular file (136, about 1 MB each) is
streamed whole: its units, the region of its electrode group and its trial table. The 37
intracellular files (behavior+icephys) are from a different experiment of Guo et al. 2017 and hold
no units; only their metadata is read.

In [ ]:
assets = tc.list_assets()
draft = requests.get(f"{tc.API}/dandisets/{tc.DANDISET}/versions/draft/assets/?page_size=1000", timeout=60).json()
draft_blobs = {a["path"]: a["blob"] for a in draft["results"]}
same = sum(draft_blobs.get(a["path"]) == a["blob"] for a in assets)
print(f"{len(assets)} assets in {tc.VERSION}; {same} have the same blob in the draft ({len(draft_blobs)} draft assets)")
ecephys = sorted([a for a in assets if "ecephys" in a["path"]], key=lambda a: a["path"])
icephys = [a for a in assets if "icephys" in a["path"]]
print(len(ecephys), "extracellular files,", len(icephys), "intracellular files,",
      f"{sum(a['size'] for a in icephys) / 1e9:.1f} GB of intracellular data (not read)")
t0 = time.time()
allsess = tc.read_sessions(ecephys, workers=12)
print(f"streamed {len(allsess)} files in {time.time() - t0:.0f} s")

In [ ]:
# One file in full: every group and every column of every table.
s0 = allsess[0]
import h5py, remfile
f = h5py.File(remfile.File(tc._s3_url(ecephys[0]["asset_id"])), "r")
def walk(g, pre=""):
    for k in g:
        o = g[k]
        if isinstance(o, h5py.Dataset):
            print(f"{pre}/{k}", o.shape, o.dtype)
        else:
            walk(o, f"{pre}/{k}")
walk(f)
print("units columns:", [c.decode() for c in f["units"].attrs["colnames"]])
print("trials columns:", [c.decode() for c in f["intervals/trials"].attrs["colnames"]])
for c in f["intervals/trials"].attrs["colnames"]:
    print(f"  trials/{c.decode()}: {f['intervals/trials'][c].attrs.get('description', '')}")
f.close()

## Data check

Per subject and session: the region named by the electrode group, the number of units, and the
trials. Then the checks of the description against the values for the quantities used: trial
outcomes and photostimulation, epoch durations (delay onset to go cue), spike-time order and
range, the recording span of each unit, coordinates, and recordings deposited twice.

In [ ]:
rows = []
for s in allsess:
    t = s["trials"]
    rows.append(dict(subject=s["subject"], session=s["path"].split("_ses-")[1][:15], region=s["region"],
                     units=len(s["spikes"]), trials=len(t["id"]),
                     correct=int((t["response"] == "correct").sum()),
                     correct_nostim=int(((t["response"] == "correct") & (t["stim_present"] == 0)).sum())))
survey = pd.DataFrame(rows)
by_subject = survey.groupby(["subject", "region"]).agg(sessions=("session", "size"), units=("units", "sum")).unstack(fill_value=0)
print(by_subject.to_string())
print()
print("regions named in the files:", Counter(survey.region))
print("subjects:", survey.subject.nunique(), "sessions:", len(survey), "units:", survey.units.sum())

In [ ]:
mouse = "anm00264943"
ms = [s for s in allsess if s["subject"] == mouse]
print(survey[survey.subject == mouse].to_string(index=False))
for s in ms:
    t = s["trials"]
    delay = t["cue_start_time"] - t["pole_out_time"]
    sample = t["pole_out_time"] - t["pole_in_time"]
    ok = np.isfinite(delay)
    sp = np.concatenate(s["spikes"])
    print(f"\n{s['path'].split('/')[1]}  region {s['region']}")
    print("  type:", dict(Counter(t["type"])), " response:", dict(Counter(t["response"])))
    print("  stim_present:", dict(Counter(t["stim_present"])), " photo_stim_period:", dict(Counter(t["photo_stim_period"])),
          " photo_stim_type:", dict(Counter(t["photo_stim_type"])))
    print("  is_good among correct trials:", dict(Counter(t["is_good"][t["response"] == "correct"])))
    print(f"  delay (go cue - pole out) s: min {np.nanmin(delay):.2f} median {np.nanmedian(delay):.2f} max {np.nanmax(delay):.2f};"
          f" sample (pole out - pole in) s: min {np.nanmin(sample):.2f} median {np.nanmedian(sample):.2f} max {np.nanmax(sample):.2f};"
          f" trials without epoch times: {(~ok).sum()} (types {dict(Counter(t['type'][~ok]))})")
    print(f"  spike times {sp.min():.1f} to {sp.max():.1f} s; trials {t['start_time'].min():.1f} to {t['stop_time'].max():.1f} s;"
          f" spike trains sorted: {all(np.all(np.diff(x) >= 0) for x in s['spikes'])}")
    cov = np.array([(x[-1] - x[0]) for x in s["spikes"]]) / (np.nanmax(t["pole_out_time"]) - np.nanmin(t["pole_out_time"]))
    print(f"  unit recording span (first to last spike) as a fraction of the trial period: median {np.median(cov):.2f},"
          f" range {cov.min():.2f} to {cov.max():.2f}")
    print("  cell_type:", dict(Counter(s["units"]["cell_type"])), " unit_z values:", np.unique(np.round(s["units"]["unit_z"], 3)))
    print("  electrode group location:", s["location"])

In [ ]:
# Coordinates: the electrode-group location and the electrode positions are the same in every file.
locs = Counter(l.split("coordinate_ref")[1] for s in allsess for l in s["location"])
print("coordinate part of the electrode group location, over all files:", dict(locs))

# Recordings deposited twice: identical spike trains, and identical trial tables, across all 136 files.
h_unit, h_trial = {}, {}
for s in allsess:
    for u, x in enumerate(s["spikes"]):
        h_unit.setdefault(hashlib.sha1(np.round(x, 6).tobytes()).hexdigest(), []).append((s["path"], u))
    key = hashlib.sha1(np.round(np.nan_to_num(s["trials"]["pole_out_time"]), 4).tobytes()).hexdigest()
    h_trial.setdefault(key, []).append(s["path"])
dup_units = [v for v in h_unit.values() if len(v) > 1]
dup_trials = [v for v in h_trial.values() if len(v) > 1]
print(f"{sum(len(s['spikes']) for s in allsess)} spike trains; identical pairs: {len(dup_units)}")
for v in dup_units[:10]:
    print("  ", v)
print(f"identical trial tables (delay onsets): {len(dup_trials)}")
for v in dup_trials[:10]:
    print("  ", v)
# Within the paper's mouse, near-identical trains (same spike count and first spike) across sessions:
sig = Counter((len(x), round(float(x[0]), 3)) for s in ms for x in s["spikes"])
print("units of", mouse, "sharing spike count and first spike time:", sum(v > 1 for v in sig.values()))

### Clock

The trial table and the spikes must share a clock. Choice (lick direction of correct trials without
photostimulation) is decoded from the spike counts of each session's units in 200 ms windows around
delay onset, with a cross-validated nearest-class-mean decoder, and compared with the same decoder
on shuffled choices. ALM and thalamic activity in this task should carry little choice
information before the sample epoch (before -1.3 s), more during the sample and delay epochs, and
most after the go cue (1.3 s). A clock offset of more than a few hundred ms would move or remove
this rise.

In [ ]:
edges = np.round(np.arange(-2.6, 2.61, 0.2), 2)
rng_clock = np.random.default_rng(tc.seed("clock-check"))
clock = []
for s in ms:
    idx, ch, on = tc.select_trials(s, "T1")
    acc = tc.choice_decoding(s, on, ch, edges, rng_clock)
    null = np.mean([tc.choice_decoding(s, on, rng_clock.permutation(ch), edges, rng_clock) for _ in range(5)], 0)
    clock.append(acc)
    pre = acc[edges[:-1] < -1.4].mean(); dl = acc[(edges[:-1] >= 0) & (edges[:-1] < 1.3)].mean()
    rsp = acc[edges[:-1] >= 1.4].mean()
    print(f"{s['region']:8s} {s['path'].split('_ses-')[1][:15]}  n={len(ch)}  accuracy before sample {pre:.2f}, delay {dl:.2f}, "
          f"after go cue {rsp:.2f}; shuffled choices {null.mean():.2f} (max over windows {null.max():.2f})")
clock = np.array(clock)
fig, ax = plt.subplots(figsize=(7, 3))
for a, s in zip(clock, ms):
    ax.plot(edges[:-1] + 0.1, a, color="C0" if s["region"] == "ALM" else "C1", alpha=0.7)
for x in (-1.3, 0, 1.3):
    ax.axvline(x, color="k", lw=0.5)
ax.set_xlabel("time from delay onset (s)"); ax.set_ylabel("choice decoding accuracy"); ax.set_ylim(0.3, 1.0)
ax.set_title("ALM (blue) and thalamus (orange) sessions of " + mouse)
plt.show()
print(f"plotted accuracy ranges from {clock.min():.2f} to {clock.max():.2f}; none outside the axes")

## wm-sessions-link

> The dandiset holds the working memory data the paper reanalyzed, five sessions from one mouse with
> 151 ALM units and 72 thalamic units in total.

Criterion: among the mice with ALM and thalamic units, find one with five sessions whose unit totals
are within tolerance (151 +/- 15 ALM, 72 +/- 7 thalamic) under a reading of W1. All sorted units
are counted (the files carry no isolation-quality label; `cell_type` is a spike-width class).

In [ ]:
both = by_subject[(by_subject[("units", "ALM")] > 0) & (by_subject[("units", "Thalamus")] > 0)]
print("mice with both ALM and thalamic units:")
print(both.to_string())
fit = []
for subj, r in both.iterrows():
    alm_n, th_n = r[("units", "ALM")], r[("units", "Thalamus")]
    ok = abs(alm_n - 151) <= 15 and abs(th_n - 72) <= 7
    fit.append(ok)
    print(f"{subj}: ALM {alm_n} units in {r[('sessions', 'ALM')]} sessions, thalamus {th_n} units in "
          f"{r[('sessions', 'Thalamus')]} sessions; within tolerance: {ok}")
print("mice that fit:", int(np.sum(fit)))
# Other regions named in the files, for completeness:
print("mice with SNr units:", sorted(survey[survey.region == "SNr"].subject.unique()))
alm_sessions = [s for s in ms if s["region"] == "ALM"]
thal_sessions = [s for s in ms if s["region"] == "Thalamus"]
print(f"\n{mouse}: {len(alm_sessions)} ALM sessions with {sum(len(s['spikes']) for s in alm_sessions)} units "
      f"({[len(s['spikes']) for s in alm_sessions]}); {len(thal_sessions)} thalamic sessions with "
      f"{sum(len(s['spikes']) for s in thal_sessions)} units ({[len(s['spikes']) for s in thal_sessions]}); "
      f"sessions holding both: 0")

**Result: reproduced.** One mouse (anm00264943) fits: its five ALM sessions hold 151 units and its
thalamic sessions hold 72 units, exactly the paper's totals; the only other mouse with ALM units
besides thalamic-labelled ones (anm00261335) has SNr, not thalamic, units. The thalamic units come
from six further sessions, so the mouse has eleven sessions in all, none with both regions; the
paper's "5 behavioral sessions" matches the ALM sessions (Figure 6C, n = 5). All sorted units are
counted.

## Preparing the paper's mouse

Correct trials, under reading T1 (no photostimulation) and T2 (all correct trials), aligned to delay
onset, binned at 10 ms and smoothed at each sigma of the grid (helper docstring).

In [ ]:
t0 = time.time()
PREP = {r: [tc.Prepared(s, r) for s in ms] for r in ("T1", "T2")}
ALM = {r: [p for p in PREP[r] if p.region == "ALM"] for r in PREP}
THAL = {r: [p for p in PREP[r] if p.region == "Thalamus"] for r in PREP}
for r in PREP:
    print(r, "trials per session (left, right):",
          [(int((p.choice == 0).sum()), int((p.choice == 1).sum())) for p in PREP[r]])
    for p in PREP[r]:
        x = p.rates[50]
        assert np.isfinite(x).all()
    print(f"  rates at sigma 50 ms: {min(p.rates[50].min() for p in PREP[r]):.1f} to "
          f"{max(p.rates[50].max() for p in PREP[r]):.1f} spikes/s;"
          f" fraction of unit-trial pairs inside the unit's span: "
          f"{np.mean(np.concatenate([p.span.ravel() for p in PREP[r]])):.2f}")
print(f"prepared in {time.time() - t0:.0f} s; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")

## fig6-wm-choice-r2

> In the working memory task, ALM population activity explains the animal's choice well (R2 = 0.97).

Criterion: test-set R2 of c(t) predicted from the test-set h(t) with the training-set weights,
under R1 (per session, mean over the five sessions) and R2 (pooled 151 units subsampled to 53, mean
over 20 subsamples), trial readings T1 and T2, mean over 10 splits. Reproduced when within 0.97 +/-
0.097 under one reading. The unit-span reading U2 and the training-set R2 are shown as variants;
neither decides the outcome.

In [ ]:
CL = "fig6-wm-choice-r2"
choice_res = {}
for off in (0, 1000, 2000, 3000, 4000):
    for r in ("T1", "T2"):
        for span in ((False, True) if off == 0 else (False,)):
            rng = np.random.default_rng(tc.seed(CL, off))
            r1 = tc.readout_per_session(ALM[r], rng, span=span)
            r2p = tc.readout_pooled(ALM[r], rng, span=span)
            choice_res[(off, r, span)] = (r1, r2p)
rows = []
for (off, r, span), (r1, r2p) in choice_res.items():
    rows.append(dict(seed_offset=off, trials=r, units="U2" if span else "U1",
                     R1_test_mean=np.mean([d["test"].mean() for d in r1]),
                     R1_test_median=np.median(np.concatenate([d["test"] for d in r1])),
                     R1_one_split=np.mean([d["test"][0] for d in r1]),
                     R1_train=np.mean([d["train"].mean() for d in r1]),
                     R2_test_mean=r2p["test"].mean(), R2_test_median=np.median(r2p["test"]),
                     R2_one_split=r2p["test"][:tc.N_SUB].mean(), R2_train=r2p["train"].mean()))
choice_tab = pd.DataFrame(rows)
print(choice_tab.round(3).to_string(index=False))

In [ ]:
r1, r2p = choice_res[(0, "T1", False)]
print("Claim seed, T1, U1. Per session (R1):")
for d in r1:
    print(f"  {d['path'].split('_ses-')[1][:15]}  units {d['n'][0]}  test R2 mean {d['test'].mean():.2f} median {np.median(d['test']):.2f}"
          f" (range {d['test'].min():.2f} to {d['test'].max():.2f}); validation {d['val'].mean():.2f}; training {d['train'].mean():.3f};"
          f" sigma chosen {dict(Counter(d['sigma']))}")
print(f"Pooled (R2): sigma chosen {dict(Counter(r2p['sigma']))}; fraction of fits with test R2 > 0: {(r2p['test'] > 0).mean():.2f}")
main = choice_tab[(choice_tab.seed_offset == 0) & (choice_tab.units == "U1")]
best = max(main[["R1_test_mean", "R2_test_mean"]].max())
print(f"\nlargest test R2 over the four readings (R1/R2 x T1/T2), claim seed: {best:.3f}; tolerance band 0.873 to 1")
u1 = choice_tab[choice_tab.units == "U1"]
print(f"over five seeds, readings R1/R2 x T1/T2: test R2 means range {u1[['R1_test_mean', 'R2_test_mean']].min().min():.2f} to "
      f"{u1[['R1_test_mean', 'R2_test_mean']].max().max():.2f}; medians {u1[['R1_test_median', 'R2_test_median']].min().min():.2f} to "
      f"{u1[['R1_test_median', 'R2_test_median']].max().max():.2f}")

**Result: not reproduced.** Under every reading the test-set R2 of the choice readout is far below
0.97: under the claim seed the largest of the four means (R1 or R2, T1 or T2) is -0.247, and the
medians run from -0.86 to 0.34; over five seeds no mean exceeds 0.10. The readout fits the training
averages almost perfectly (training R2 0.95 to 1.00), so the OLS readout on 18 to 53 units over 260
time points does not generalize to the test averages of 4 to 13 trials per choice. The U2 variant
gives at most 0.506. The training-set R2 is close to the paper's 0.97, which suggests the paper's
value may be an in-sample fit; this is a question for the paper's authors, not a reading of the
claim.

## fig6c-wm-leading-pcs-drive-choice

> In the working memory task, the leading PCs of ALM activity have a large influence on the readout
> of choice.

Criterion: alignment |w . u_i| / ||w|| of the readout with PCs 1 to 16, per session with the mean
over the five sessions (R1), and on the pooled subsampled population (R2). Readings (a) the PC with
the largest mean alignment is PC 1 or 2; (b) the mean alignment over PCs 1 to 3 exceeds the mean
over PCs 4 to 16. Reproduced when both hold under one trial reading, partially when one holds.

In [ ]:
CL = "fig6c-wm-leading-pcs-drive-choice"
pc_rows, profiles = [], {}
for off in (0, 1000, 2000, 3000, 4000):
    for r in ("T1", "T2"):
        for span in ((False, True) if off == 0 else (False,)):
            rng = np.random.default_rng(tc.seed(CL, off))
            r1 = tc.readout_per_session(ALM[r], rng, span=span)
            r2p = tc.readout_pooled(ALM[r], rng, span=span)
            for name, prof, sem in (("R1", np.nanmean([np.nanmean(d["align"], 0) for d in r1], 0),
                                     stats.sem([np.nanmean(d["align"], 0) for d in r1], 0, nan_policy="omit")),
                                    ("R2", np.nanmean(r2p["align"], 0), np.nanstd(r2p["align"], 0) / np.sqrt(len(r2p["align"])))):
                profiles[(off, r, span, name)] = (prof, sem)
                a = int(np.argmax(prof)) + 1
                b = prof[:3].mean() > prof[3:].mean()
                pc_rows.append(dict(seed_offset=off, trials=r, units="U2" if span else "U1", readout=name,
                                    argmax_PC=a, mean_PC1_3=prof[:3].mean(), mean_PC4_16=prof[3:].mean(),
                                    a_holds=a <= 2, b_holds=bool(b)))
pc_tab = pd.DataFrame(pc_rows)
print(pc_tab.round(3).to_string(index=False))

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(9, 3), sharey=True)
lo, hi = np.inf, -np.inf
for ax, name in zip(axs, ("R1", "R2")):
    for r, c in (("T1", "C0"), ("T2", "C2")):
        prof, sem = profiles[(0, r, False, name)]
        ax.errorbar(np.arange(1, 17), prof, sem, color=c, label=r, marker="o", ms=3)
        lo, hi = min(lo, (prof - sem).min()), max(hi, (prof + sem).max())
    ax.set_title(f"readout {name}, claim seed, U1"); ax.set_xlabel("PC"); ax.legend()
axs[0].set_ylabel("|w . u_i| / ||w||")
plt.show()
print(f"plotted range {lo:.3f} to {hi:.3f}")
for name in ("R1", "R2"):
    for r in ("T1", "T2"):
        prof, sem = profiles[(0, r, False, name)]
        print(name, r, "alignment PCs 1-16:", np.round(prof, 3))
m = pc_tab[(pc_tab.seed_offset == 0) & (pc_tab.units == "U1")]
print("\nclaim seed, U1: readings where (a) holds:", m[m.a_holds][["trials", "readout"]].values.tolist(),
      "; where (b) holds:", m[m.b_holds][["trials", "readout"]].values.tolist(),
      "; where both hold:", m[m.a_holds & m.b_holds][["trials", "readout"]].values.tolist())
s5 = pc_tab[pc_tab.units == "U1"].groupby("seed_offset").apply(lambda g: bool((g.a_holds & g.b_holds).any()))
print("both hold under some reading, per seed offset:", s5.to_dict())
s5a = pc_tab[pc_tab.units == "U1"].groupby("seed_offset").apply(lambda g: bool((g.a_holds | g.b_holds).any()))
print("at least one holds under some reading, per seed offset:", s5a.to_dict())
# Reference: alignment |w . u| / ||w|| of a random direction w with a fixed unit vector u, in d dimensions.
rng_ref = np.random.default_rng(tc.seed(CL, 1))
for d in sorted({p.n_units for p in ALM["T1"]} | {tc.N_KEEP}):
    x = rng_ref.standard_normal((20000, d))
    print(f"random-direction reference, d = {d}: mean alignment {np.mean(np.abs(x[:, 0]) / np.linalg.norm(x, axis=1)):.3f}")

**Result: reproduced, under reading R2 only.** With the pooled population subsampled to 53 units
(R2), both readings of "leading" hold for T1 and T2 under the claim seed: the largest alignment is
with PC 2 (T1) or PC 1 (T2), and the mean over PCs 1 to 3 exceeds the mean over PCs 4 to 16 (0.081
against 0.063 for T1). Both hold under some reading for all five seeds. Per session (R1, the
reading of Figure 6C, n = 5) reading (a) fails for both trial readings, with the largest alignment
at PC 16, and (b) holds only for T1 (0.107 against 0.107). All alignments are small: in 53
dimensions a random direction has a mean alignment of about 0.11 with any fixed direction, and the
R2 profile lies between 0.046 and 0.11. The readout does not generalize to test data
(fig6-wm-choice-r2), so these alignments describe an overfit readout.

## Thalamic regressions (figs6d-wm-thalamic-r2, fig6d-wm-top5-pcs, figs6f-ct-alignment-not-significant)

Each thalamic unit's trial-averaged r(t) (its own session's correct trials, split 80/10/10) is
regressed on the trial-averaged ALM population of the five ALM sessions (W9): P1, the 151 pooled
units subsampled to 53, 20 subsamples per split; P2, all 151. P3 (each thalamic unit with the ALM
units of its own session) does not apply: no session holds both. Each claim runs with its own seed;
under the claim seed every reading (T1/T2, P1/P2, U1/U2) is run, and under the four further seeds
the readings T1/T2 x P1/P2 with U1.

In [ ]:
def run_thal(claim):
    out = {}
    for off in (0, 1000, 2000, 3000, 4000):
        for r in ("T1", "T2"):
            for pool in ("P1", "P2"):
                for span in ((False, True) if off == 0 else (False,)):
                    rng = np.random.default_rng(tc.seed(claim, off))
                    out[(off, r, pool, span)] = tc.thalamic_analysis(ALM[r], THAL[r], rng, pool=pool, span=span)
    return out

t0 = time.time()
TH = {c: run_thal(c) for c in ("figs6d-wm-thalamic-r2", "fig6d-wm-top5-pcs", "figs6f-ct-alignment-not-significant")}
print(f"thalamic regressions in {time.time() - t0:.0f} s; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")

### figs6d-wm-thalamic-r2

> ALM activity explains 0.70 +/- 0.01 of the variance of VM/VAL thalamic neurons in the working
> memory task.

Criterion: test-set R2 from all cortical units of the reading of W9 (53 units in P1), mean over the
72 thalamic units; reproduced when within 0.70 +/- 0.01 under one combination of T and P.

In [ ]:
res = TH["figs6d-wm-thalamic-r2"]
rows = []
for (off, r, pool, span), d in res.items():
    rows.append(dict(seed_offset=off, trials=r, pool=pool, units="U2" if span else "U1",
                     mean_test_R2=np.nanmean(d["full"]), median_test_R2=np.nanmedian(d["full"]),
                     sem=stats.sem(d["full"], nan_policy="omit"), n_units=int(np.isfinite(d["full"]).sum()),
                     frac_positive=np.nanmean(d["full"] > 0), min_fits=int(d["n_fits"].min()),
                     mean_train_R2=np.nanmean(d["full_train"])))
s6d = pd.DataFrame(rows)
print(s6d.round(3).to_string(index=False))
m = s6d[(s6d.seed_offset == 0) & (s6d.units == "U1")]
print("\nclaim seed, U1: combinations within 0.69 to 0.71:", m[(m.mean_test_R2 - 0.70).abs() <= 0.01][["trials", "pool"]].values.tolist())
print(f"test R2 means over all seeds and U1 readings: {s6d[s6d.units == 'U1'].mean_test_R2.min():.2f} to {s6d[s6d.units == 'U1'].mean_test_R2.max():.2f}")
d = res[(0, "T1", "P1", False)]
fig, ax = plt.subplots(figsize=(5, 3))
x = np.sort(d["full"]); ax.plot(np.clip(x, -5, 1), np.arange(1, len(x) + 1) / len(x))
ax.set_xlabel("test R2 from all cortical units (clipped at -5)"); ax.set_ylabel("cumulative fraction")
plt.show()
print(f"T1 P1 U1: test R2 range {x.min():.2f} to {x.max():.2f}; {np.sum(x < -5)} units below -5 drawn at -5")

**Result: not reproduced.** No combination of trial and pooling readings gives a mean test-set R2
near 0.70: under the claim seed the means run from -5.16 to -1.91, and no unit has a positive test
R2 under P1. The training-set R2 of the same regressions, a diagnostic only, is 0.73 to 0.77 under
P1 over the five seeds and near 1 under P2.

### fig6d-wm-top5-pcs

> In the working memory task, the top 5 ALM PCs capture 0.63 +/- 0.04 of the explainable variance of
> thalamic neurons.

Criterion: mean normalized R2(5) over the 72 thalamic units under N1 (R2(5)/R2(16)), N2 (R2(5)/R2
from all cortical units) or N3 (each unit's curve rescaled to 0 to 1); reproduced when within 0.63
+/- 0.04 under one combination of T, P and N. Normalization is applied to each unit's mean curve
over splits and subsamples.

In [ ]:
res = TH["fig6d-wm-top5-pcs"]
rows = []
for (off, r, pool, span), d in res.items():
    for how in ("N1", "N2", "N3"):
        v = tc.normalized(d["r2k"], d["full"], how)[4]
        vt = tc.normalized(d["r2k_train"], d["full_train"], how)[4]
        rows.append(dict(seed_offset=off, trials=r, pool=pool, units="U2" if span else "U1", norm=how,
                         mean=np.nanmean(v), median=np.nanmedian(v), sem=stats.sem(v, nan_policy="omit"),
                         n=int(np.isfinite(v).sum()), mean_train=np.nanmean(vt)))
t5 = pd.DataFrame(rows)
print(t5.round(3).to_string(index=False))
m = t5[(t5.seed_offset == 0) & (t5.units == "U1")]
hit = m[(m["mean"] - 0.63).abs() <= 0.04]
print("\nclaim seed, U1: combinations within 0.59 to 0.67:", hit[["trials", "pool", "norm"]].values.tolist())
for s, g in t5[t5.units == "U1"].groupby("seed_offset"):
    print(f"seed offset {s}: combinations within tolerance: {g[(g['mean'] - 0.63).abs() <= 0.04][['trials', 'pool', 'norm']].values.tolist()}")
d = res[(0, "T1", "P1", False)]
print("\nT1 P1 U1, mean test R2(k), k = 1 to 16:", np.round(np.nanmean(d["r2k"], 1), 3))
print("T1 P1 U1, mean training R2(k), k = 1 to 16:", np.round(np.nanmean(d["r2k_train"], 1), 3))
print("units whose test R2(16) is positive:", int(np.sum(d["r2k"][-1] > 0)), "of", d["r2k"].shape[1])

**Result: not reproduced.** No combination of T, P and N gives a mean normalized R2(5) within 0.59
to 0.67, under any of the five seeds. The test-set R2(k) is near 0 at k = 1 and falls as k grows (the
regressions on more PCs overfit; no unit has a positive test R2(16)). Over the five seeds with U1,
N1 gives at most 0.31 under P1 and means from -1.02 to 2.01, driven by outliers, under P2; N2 gives
-0.09 to 0.11; and N3, which rescales each curve from its minimum to its maximum, gives 0.76 to
0.86. On the training set, a diagnostic only, N2 gives 0.68 and N3 0.70 under T1 and P1.

### figs6f-ct-alignment-not-significant

> For individual thalamic neurons, the difference between the alignment of corticothalamic weights
> with the leading cortical PC and with the readout direction is not significant in either dataset
> (one-sided paired t tests, p = 0.09 and p = 0.17).

Criterion: a claim of absence on the paper's test. Corticothalamic weights from the regression on
all 16 PCs mapped back to unit space; alignment |cos| with PC 1 and with the readout on the same
cortical population; one-sided paired t test over the 72 units, PC greater than readout;
reproduced when p >= 0.05. The cortical population follows the Figure S6 legend ("in all
analysis", P1); T1 is the main trial reading (W2 lists it first). The other readings and the
weights of the regression on all units are variants.

In [ ]:
res = TH["figs6f-ct-alignment-not-significant"]
rows = []
for (off, r, pool, span), d in res.items():
    for w in ("16 PCs", "all units"):
        a, b = (d["a_pc"], d["a_ro"]) if w == "16 PCs" else (d["a_pc_full"], d["a_ro_full"])
        ok = np.isfinite(a) & np.isfinite(b)
        t = stats.ttest_rel(a[ok], b[ok], alternative="greater")
        diff = a[ok] - b[ok]
        rows.append(dict(seed_offset=off, trials=r, pool=pool, units="U2" if span else "U1", weights=w,
                         mean_align_PC1=a[ok].mean(), mean_align_readout=b[ok].mean(),
                         mean_diff=diff.mean(), cohen_dz=diff.mean() / diff.std(ddof=1), t=t.statistic, p=t.pvalue, n=int(ok.sum())))
s6f = pd.DataFrame(rows)
print(s6f.round(4).to_string(index=False))
main = s6f[(s6f.seed_offset == 0) & (s6f.trials == "T1") & (s6f.pool == "P1") & (s6f.units == "U1") & (s6f.weights == "16 PCs")].iloc[0]
print(f"\nmain reading (claim seed, T1, P1, U1, 16 PCs): PC1 {main.mean_align_PC1:.3f}, readout {main.mean_align_readout:.3f}, "
      f"difference {main.mean_diff:.3f}, t = {main.t:.2f}, one-sided p = {main.p:.2g}, n = {main.n}")
sd = s6f[(s6f.trials == "T1") & (s6f.pool == "P1") & (s6f.units == "U1") & (s6f.weights == "16 PCs")]
print("main reading over five seeds, p:", sd.set_index("seed_offset").p.round(4).to_dict())
print("readings with p < 0.05 (claim seed):", s6f[(s6f.seed_offset == 0) & (s6f.p < 0.05)][["trials", "pool", "units", "weights"]].values.tolist())
print("readings with p >= 0.05 (any seed):", s6f[s6f.p >= 0.05][["seed_offset", "trials", "pool", "units", "weights"]].values.tolist())
rng_ref = np.random.default_rng(tc.seed("figs6f-ct-alignment-not-significant", 1))
for d in (16, tc.N_KEEP, 151):
    x = rng_ref.standard_normal((20000, d))
    print(f"random-direction reference, d = {d}: mean alignment {np.mean(np.abs(x[:, 0]) / np.linalg.norm(x, axis=1)):.3f}")

**Result: not reproduced.** On the main reading (T1, P1, U1, weights from the 16 PCs) the
corticothalamic weights are more aligned with PC 1 than with the readout (0.094 against 0.071, mean
difference 0.023, t = 4.62, one-sided p = 8.3e-06, n = 72), under all five seeds, so the paper's test
finds the difference the claim says it does not. Every reading with the 16-PC weights gives p < 0.05
under the claim seed; only T2 with P1 gives p >= 0.05 under two of the other seeds. With the weights
of the regression on all units the readout alignment is the larger one (one-sided p = 1). Both
alignments are below the random-direction references printed above (about 0.2 for the 16-PC span,
0.11 for 53 units), and the regressions behind the weights do not predict test data
(figs6d-wm-thalamic-r2).

## The simulation claims

The 24 claims on the paper's models (Figures 1 to 5, S1 to S5) need the models run as the paper
ran them: 10,000 learning trials of 200 time steps for each of 40 initializations, for every
network size and condition, and, for the meta-learning claims, 10,000 epochs of 200 such trials
each. One epoch of the paper's own motor control model (`thalamocortical__motor.py`, commit 2aa94dd,
reimplemented in `tcloop.rflo_motor`) is timed here at the smallest and largest thalamus.

In [ ]:
rng = np.random.default_rng(tc.seed("fig1c-tc-learning-improves-with-thalamus"))
per_epoch = {}
for M in (1, 8, 32, 128):
    t0 = time.time(); e = tc.rflo_motor(3, M, rng); per_epoch[M] = (time.time() - t0) / 3
    print(f"M = {M}: {per_epoch[M] * 1000:.1f} ms per learning trial; errors of the 3 trials {np.round(e, 3)}")
Ms = [1, 2, 4, 8, 16, 32, 64, 128]
interp = np.interp(np.log2(Ms), np.log2(list(per_epoch)), list(per_epoch.values()))
one_curve = interp.sum() * 10000 * 40
print(f"\nfig1c/fig1d: M = 1 to 128, 10,000 trials, 40 runs: {one_curve / 3600:.1f} h on one core, "
      f"{one_curve / 3600 / 4:.1f} h on four")
m1 = per_epoch[1] * 10000 * 40
print(f"fig3ab: M = 1, 40 runs, one structure and one task: {m1 / 3600:.1f} h; three structures and two tasks: {6 * m1 / 3600:.1f} h")
print(f"fig2d: meta-learning, 10,000 epochs of 200 trials at M = 32 for one run: "
      f"{np.interp(5, np.log2(list(per_epoch)), list(per_epoch.values())) * 200 * 10000 / 3600:.0f} h before backpropagation")
print("notebook limit: 1 h")

**Result: blocked, too large.** One set of 40 runs of 10,000 learning trials at the smallest
thalamus (M = 1) takes about 0.5 h on one core, and every simulation claim compares at least two
such sets (fig3ab needs six, about 3 h; the curves of Figure 1 need about 8 h on one core, 2 h on
four),
over the one-hour limit of a notebook. The meta-learning claims need about 5 h of forward
simulation for a single run before any backpropagation.

## Summary

| Claim | Outcome |
|---|---|
| wm-sessions-link | reproduced: mouse anm00264943, 151 ALM units in 5 sessions, 72 thalamic units in 6 other sessions |
| fig6-wm-choice-r2 | not reproduced: test R2 at most -0.247 under the claim seed (0.97 expected) |
| fig6c-wm-leading-pcs-drive-choice | reproduced under reading R2 (pooled, subsampled); not under R1 (per session) |
| figs6d-wm-thalamic-r2 | not reproduced: mean test R2 -5.16 to -1.91 (0.70 expected) |
| fig6d-wm-top5-pcs | not reproduced: no reading within 0.59 to 0.67 |
| figs6f-ct-alignment-not-significant | not reproduced: PC 1 alignment larger than readout alignment, one-sided p = 8.3e-06 |
| Six motor-task claims | blocked: data missing |
| 23 simulation claims | blocked: too large |
| figs5cd-thalamic-recurrence | blocked: methods underspecified |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's. The smoothing grid, the 10 ms bin and the padding are
standard choices the paper does not give. Sigma is chosen per thalamic unit on the validation R2 of
the 16-PC regression, and the normalizations are applied to each unit's mean curve over splits and
subsamples. P3 does not apply, as no session holds both ALM and thalamic units. The thalamic units
are regressed on ALM activity from other sessions, as the paper's design implies; their test sets
are their own sessions' trials.

What was found about the dataset. The paper's mouse is anm00264943, and its unit totals match the
paper's exactly; the thalamic units come from six sessions, not the five the paper states. The
files give no interval over which each unit was recorded, though many units fire in only part of
their session. The electrode group location gives the same coordinates (AP 2.50, ML -1.50, DV
0.00) in all 137 files, thalamic and SNr recordings included, and the thalamic recordings name no
nucleus. `photo_stim_type` is "stimulation" on every trial, including those without
photostimulation; `stim_present` and `photo_stim_period` carry the distinction. No spike train or
trial table is deposited twice, and choice decoding around delay onset is consistent with spikes
and trials sharing a clock in all eleven sessions.

In [ ]:
print(f"total time {(time.time() - started) / 60:.1f} min; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")